# 03 · Getting real data with yfinance 🌐

Time to download **real stock-market data**. The `yfinance` library fetches price
history from Yahoo Finance and hands it back as a pandas DataFrame — the exact
table type you just practised on.

> ⚠️ **This notebook needs the internet.** If a download fails, check your
> connection and try again. Yahoo occasionally rate-limits — if so, wait a moment
> and re-run the cell.

## What you'll learn
- `import yfinance as yf`
- The **canonical download** we'll use everywhere (and why its options matter)
- Inspecting the result with `.head()`, `.shape`, `.columns`
- What each column (`Open`, `High`, `Low`, `Close`, `Volume`) means
- Trying a different ticker and time period
- Saving your data to a CSV file
- One company vs many — **multi-level columns**

## 1. Import yfinance

In [ ]:
import yfinance as yf

## 2. Download some data (the form we'll always use)

`yf.download(...)` returns a DataFrame with **one row per trading day**. We pass a
few options every time so the table comes back tidy and easy to work with. Let's
grab six months of Apple.

In [ ]:
data = yf.download("AAPL", period="6mo", auto_adjust=True, multi_level_index=False, progress=False)
data.head()

### What those options mean

We'll use this **exact form** for every single-company download from now on. Here's
what each option does, in plain English:

- **`auto_adjust=True`** — tidy the prices for stock splits and dividends so the
  history lines up cleanly. You get one ready-to-use `Close` column (no separate
  "Adj Close").
- **`multi_level_index=False`** — give us **plain column names** (`Close`, `Open`,
  ...). This matters a lot: it means `data["Close"]` is a **single column** (a
  Series), which is exactly what our filtering and charts expect. Without it,
  yfinance stacks the ticker name on top of each column and `data["Close"]` comes
  back as a mini-table instead — confusing for now.
- **`progress=False`** — don't print a download progress bar; keeps the output
  clean.

Common periods you can pass: `"1mo"`, `"6mo"`, `"1y"`, `"5y"`, `"max"`.

## 3. Inspect it

Same tools as notebook 02 — because it's the same kind of object, a DataFrame.

In [ ]:
print("Shape (rows, cols):", data.shape)     # ~126 rows for 6 months
print("Columns:", list(data.columns))

In [ ]:
data.tail()          # the most recent few days

## 4. What the columns mean

| Column | Meaning |
| ------ | ------- |
| `Open` | Price when the market opened that day |
| `High` | Highest price during the day |
| `Low` | Lowest price during the day |
| `Close` | Price when the market closed — the "headline" price |
| `Volume` | How many shares changed hands that day |

The **date** is the row label (the **index**), not a normal column. We mostly
care about `Close`. Because we passed `multi_level_index=False`, `data["Close"]`
is a single column (a Series) you can compare and chart directly.

In [ ]:
data["Close"].tail()       # just the closing prices, most recent first at the end

In [ ]:
print("Highest close in the period:", data["Close"].max())
print("Lowest close in the period: ", data["Close"].min())
print("Average close:              ", data["Close"].mean())

### 🏋️ Your turn

Download **6 months of Microsoft** (`"MSFT"`) into a variable called `msft` — use
the **same options** as above — then show its `.head()`.

In [ ]:
# TODO: download MSFT for period "6mo" with auto_adjust=True, multi_level_index=False,
# progress=False into `msft`, then call msft.head().

**One way to do it:**

```python
msft = yf.download("MSFT", period="6mo", auto_adjust=True, multi_level_index=False, progress=False)
msft.head()
```

## 5. Try a different period

The `period` argument changes how far back you go. Here's a full year of Tesla —
same options, just a longer period.

In [ ]:
tsla = yf.download("TSLA", period="1y", auto_adjust=True, multi_level_index=False, progress=False)
print(tsla.shape)          # a year -> ~250 trading days
tsla.head()

### 🏋️ Your turn

Download just **1 month** (`"1mo"`) of Apple and print how many rows you got with
`.shape`. (Roughly 21 trading days in a month — weekends and holidays don't count.)

In [ ]:
# TODO: download AAPL for period "1mo" (same options) and print its .shape.

**One way to do it:**

```python
aapl_1mo = yf.download("AAPL", period="1mo", auto_adjust=True, multi_level_index=False, progress=False)
print(aapl_1mo.shape)
```

## 6. Save it to a file

Downloading every time is slow and needs internet. Save a DataFrame to a **CSV**
(a simple spreadsheet file) so you can reload it instantly later. The project has
a `data/` folder ready for this.

In [ ]:
data.to_csv("../data/aapl.csv")
print("Saved! Look in the data/ folder for aapl.csv")

To read it back later you'd use `pd.read_csv("../data/aapl.csv", index_col=0,
parse_dates=True)` — `index_col=0` keeps the date as the index, and
`parse_dates=True` reads it back as real dates.

## 7. One company vs many (multi-level columns) 🪜

So far we've asked for **one** company and used `multi_level_index=False` to get
flat column names — so `data["Close"]` is a single column (a Series). Start there:
it's the simple case, and it's all you need for most of the dashboard.

But what if you want to compare **several** companies at once? Pass a **list** of
tickers. Now yfinance can't use plain column names — `Close` alone would be
ambiguous (whose close?) — so it gives you **two-level** columns like
`('Close', 'AAPL')`: the measurement on top, the ticker underneath.

In [ ]:
many = yf.download(["AAPL", "MSFT"], period="6mo", auto_adjust=True, progress=False)
many.head()

See how the column headers now have **two rows**? The top level is the measurement
(`Close`, `Open`, ...) and the second level is the ticker (`AAPL`, `MSFT`).

You dig in one level at a time:

- `many["Close"]` → **all the closes**, one column per company.
- `many["Close"]["AAPL"]` → **just Apple's** closes (back to a single Series).

In [ ]:
many["Close"].head()          # every company's closing prices, side by side

In [ ]:
many["Close"]["AAPL"].head()  # drill down to just Apple's closes -> a Series

### 🏋️ Your turn

From `many`, pull out **just Microsoft's** closing prices and show the first few.

In [ ]:
# TODO: from `many`, select MSFT's closes (Close, then MSFT) and call .head().

**One way to do it:**

```python
many["Close"]["MSFT"].head()
```

> 🧭 **The plan:** start simple with **one** company and flat columns
> (`multi_level_index=False`) — that's what the dashboard uses. You'll meet these
> multi-level columns only when you **compare several companies**, which is one of
> the **stretch goals** in [`../SPEC.md`](../SPEC.md). File the idea away for now.

## 🎉 You're pulling live data!

You can now fetch real prices for any ticker and any period, inspect the table,
and save it — and you know the difference between one company (flat columns) and
several (multi-level columns). This is exactly what **Stage 1** of the dashboard
does: `yf.download(...)` then show the table.

**Next up:** [`04_filtering_and_searching.ipynb`](04_filtering_and_searching.ipynb)
— the heart of the project: filtering and searching your data.

📖 Companion guide:
[`../guides/03-getting-data-yfinance.md`](../guides/03-getting-data-yfinance.md)